# 🩺 Syght — Healthcare Insurance Fraud Detector Platform
## Unit-1 Experiment 4: Building Classification Models Using Different Machine Learning Algorithms
**Course Specialization**: AI & Machine Learning with Cloud & Azure Tools  
**Project**: Healthcare Insurance Fraud Detection ()  

---

### 🎯 Experiment Aim & Overview:
Systematic training, benchmarking, and selection of 7 machine learning algorithms against heuristic rule baselines using 5-fold stratified cross-validation and the 1-SE rule.

---

### 📚 Theoretical Background
In industrial machine learning, we should never assume complex algorithms are automatically superior.
Syght establishes a disciplined, scientific benchmarking protocol:
1. **Heuristic Rule Baselines**: Simple volume rules (`Rule_ClaimVolume`, `Rule_TotalReimbursed`).
2. **7 Machine Learning Candidates**: Logistic Regression (L1/L2), Random Forest, Extra Trees, Gradient Boosting, XGBoost, and LightGBM.
3. **5-Fold Stratified Cross-Validation**: Testing every candidate under identical fold splits.
4. **The 1-Standard-Error (1-SE) Rule**: Selecting the most parsimonious model whose performance is within 1 standard error of the top score.

In [ ]:
import numpy as np
import pandas as pd
from ml_pipeline.medicare_simulator import generate_simulated_medicare_data
from ml_pipeline.feature_engineering import build_provider_features
from ml_pipeline.train import get_candidate_models, evaluate_cv_candidates, select_model_1se
from ml_pipeline.config import SyghtConfig

cfg = SyghtConfig()
lbl, bene, ip, op = generate_simulated_medicare_data(n_providers=50, seed=42)
feats = build_provider_features(lbl, bene, ip, op, config=cfg)
feature_cols = [c for c in feats.columns if c not in ['Provider', 'fraud', 'exposure_amount']]

candidates = get_candidate_models(cfg)
print(f'Total Candidates to Benchmark: {len(candidates)}')
for name in candidates:
    print(f' - {name}')

### Step 1: Running 5-Fold Stratified Cross-Validation Benchmark

In [ ]:
# Run CV evaluation
cv_results, oof_preds = evaluate_cv_candidates(feats, feature_cols, candidates, config=cfg)
cv_summary = cv_results.sort_values(by='pr_auc_mean', ascending=False)

print('=== 5-FOLD STRATIFIED CV BENCHMARK LEADERBOARD ===')
print(cv_summary[['model', 'pr_auc_mean', 'pr_auc_std', 'roc_auc_mean', 'roc_auc_std', 'brier_mean']].to_string(index=False))

### Step 2: Model Selection via the 1-Standard-Error (1-SE) Rule

In [ ]:
selection = select_model_1se(cv_summary)
print(f'Champion Model Selected: {selection["selected_model"]}')
print(f'Best ML PR-AUC:          {selection["best_ml_score"]:.4f}')
print(f'Best Rule Baseline:      {selection["best_rule_model"]} ({selection["best_rule_score"]:.4f})')
print(f'Is ML Justified (>1 SE): {selection["ml_justified"]}')

### Step 3: Visualization of Model Performance Benchmark

In [ ]:
import matplotlib.pyplot as plt

models = cv_summary['model'].tolist()
means = cv_summary['pr_auc_mean'].to_numpy()
stds = cv_summary['pr_auc_std'].to_numpy()

plt.figure(figsize=(10, 6))
colors = ['#10b981' if 'LightGBM' in m or 'XGBoost' in m else '#3b82f6' if 'Rule' not in m else '#94a3b8' for m in models]
plt.barh(models[::-1], means[::-1], xerr=stds[::-1], color=colors[::-1], capsize=4, edgecolor='black', alpha=0.85)
plt.xlabel('Cross-Validation PR-AUC (Mean ± 1 Std)')
plt.title('Algorithm Comparison Benchmark with Standard Error Bars')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()